# Seleccion de Caracteristicas Multimetodo
Objetivo: comparar varios metodos de seleccion para `tipo_violencia` y `nivel_riesgo_victima`, y obtener un ranking consensuado de features.

## 1) Librerias

In [1]:
# !pip install pandas pyarrow numpy scipy scikit-learn matplotlib seaborn
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif, RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import joblib

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 180)
sns.set(style="whitegrid")


## 2) Carga de datos
Carga el parquet final y estandariza nombre de target de riesgo.

In [2]:
PARQUET_PATH = "/home/munasqa/MAESTRIA/opencode/base_modelado.parquet"
df = pd.read_parquet(PARQUET_PATH)

if "nivel_de_riesgo_victima" in df.columns and "nivel_riesgo_victima" not in df.columns:
    df = df.rename(columns={"nivel_de_riesgo_victima": "nivel_riesgo_victima"})

targets = ["tipo_violencia", "nivel_riesgo_victima"]
for t in targets:
    if t not in df.columns:
        raise ValueError(f"Falta target: {t}")

print(df.shape)
display(df[targets].head())


(932860, 131)


,tipo_violencia,nivel_riesgo_victima
0,1,1
1,0,0
2,0,0
3,2,1
4,2,1


## 3) Preparacion de features
Excluye targets y posibles fugas evidentes.

In [3]:
leakage_obvio = ["tipo_violencia", "nivel_riesgo_victima", "tipo_violencia_lbl", "nivel_riesgo_victima_lbl"]
features = [c for c in df.columns if c not in leakage_obvio]

# NO convertimos a string aqui para no saturar la RAM
print("Features candidatas:", len(features))
print("Memoria actual df:", f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Features candidatas: 129
Memoria actual df: 475.25 MB


## 4) Nota metodologica (paper)
El paper de referencia menciona enfoque multi-metodo (filters + wrappers + evolutivo). Aqui implementamos una version practica y reproducible en sklearn:
- Filter: Chi2/Cramers V y Mutual Information
- Embedded: Random Forest importance + Permutation importance
- Wrapper: RFECV
Luego se consolida un ranking por consenso.

## 5) Funciones auxiliares

In [4]:
def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.values.sum()
    r, k = tab.shape
    den = min(r - 1, k - 1)
    v = np.sqrt((chi2 / n) / den) if den > 0 else np.nan
    return v, p

def preprocess_sample(X_df):
    """Limpieza e imputacion sobre la muestra para ahorrar RAM"""
    X = X_df.copy()
    for c in X.columns:
        if str(X[c].dtype) in ["object", "category"]:
            X[c] = X[c].astype(str).fillna("desconocido")
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce")
            X[c] = X[c].fillna(X[c].median())
    return X

def to_ordinal_encoded(X_df):
    """Convierte categorias a ordinales y asegura tipos int32/float32"""
    X = X_df.copy()
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    if cat_cols:
        X[cat_cols] = enc.fit_transform(X[cat_cols]).astype("float32")

    for c in X.columns:
        if c not in cat_cols:
            X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0).astype("float32")

    return X

def rank_to_score(df_rank, score_col, feature_col="feature", descending=True):
    r = df_rank[[feature_col, score_col]].dropna().copy()
    if r.empty:
        return pd.DataFrame(columns=["feature", "score"])
    r = r.sort_values(score_col, ascending=not descending).reset_index(drop=True)
    r["score"] = np.linspace(1.0, 0.0, len(r), endpoint=False)
    return r[[feature_col, "score"]]


## 6) Seleccion multimetodo por target
Se ejecutan 5 metodos y se combina un score de consenso.

In [5]:
results = {}
SAMPLE_SIZE = 100000  # Aumentado a 100k gracias a las optimizaciones de memoria

for target in targets:
    print(f"\n=== PROCESANDO TARGET: {target} ===")
    y_full = pd.to_numeric(df[target], errors="coerce")
    mask = y_full.notna()
    y_full = y_full[mask].astype(int)
    
    # Muestreo estratificado directo de df para ahorrar RAM
    if len(y_full) > SAMPLE_SIZE:
        _, idx_sample = train_test_split(
            np.arange(len(y_full)), 
            test_size=SAMPLE_SIZE, 
            stratify=y_full, 
            random_state=42
        )
        X_use = df.loc[mask].iloc[idx_sample][features].copy()
        y = y_full.iloc[idx_sample].copy()
    else:
        X_use = df.loc[mask, features].copy()
        y = y_full.copy()
    
    gc.collect()

    # Preprocesamiento ligero solo sobre la muestra
    print("Preprocesando muestra (imputacion y limpieza)...")
    X_use = preprocess_sample(X_use)

    # 1) Cramers V
    print("Calculando Cramers V...")
    rows_cv = []
    for c in X_use.columns:
        v, p = cramers_v(X_use[c], y)
        rows_cv.append((c, v, p))
    r_cv = pd.DataFrame(rows_cv, columns=["feature", "cramers_v", "p_value"]).dropna()
    del rows_cv
    gc.collect()

    # 2) Mutual Information
    print("Calculando Mutual Information...")
    X_enc = to_ordinal_encoded(X_use)
    mi_vals = mutual_info_classif(X_enc, y, discrete_features="auto", random_state=42)
    r_mi = pd.DataFrame({"feature": X_enc.columns, "mi": mi_vals})
    
    # Split para modelos y validacion
    Xtr, Xte, ytr, yte = train_test_split(X_enc, y, test_size=0.2, stratify=y, random_state=42)

    # 3) RF Importance
    print("Entrenando Random Forest para importancias...")
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4, class_weight="balanced_subsample")
    rf.fit(Xtr, ytr)
    r_rf = pd.DataFrame({"feature": X_enc.columns, "rf_importance": rf.feature_importances_})
    gc.collect()

    # 4) Permutation importance (Sobre test set)
    print("Calculando Permutation Importance (top 40)...")
    with joblib.parallel_backend("threading"):
        perm = permutation_importance(rf, Xte, yte, n_repeats=3, random_state=42, n_jobs=2, scoring="f1_macro")
    r_perm = pd.DataFrame({"feature": X_enc.columns, "perm_importance": perm.importances_mean})
    gc.collect()

    # 5) RFECV (Wrapper sobre el top 30 de MI)
    print("Ejecutando RFECV (wrapper sobre Top 30 MI)...")
    top_for_rfe = r_mi.sort_values("mi", ascending=False).head(30)["feature"].tolist()
    X_rfe = X_enc[top_for_rfe].copy()
    rfe_est = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=2, class_weight="balanced_subsample")
    with joblib.parallel_backend("threading"):
        rfecv = RFECV(estimator=rfe_est, step=10, cv=3, scoring="f1_macro", min_features_to_select=5, n_jobs=2)
        rfecv.fit(X_rfe, y)
    r_rfe = pd.DataFrame({"feature": X_rfe.columns, "rfe_rank": rfecv.ranking_, "rfe_selected": rfecv.support_.astype(int)})
    gc.collect()

    # Consolidacion de Scores (Consenso)
    s_cv = rank_to_score(r_cv, "cramers_v", descending=True)
    s_mi = rank_to_score(r_mi, "mi", descending=True)
    s_rf = rank_to_score(r_rf, "rf_importance", descending=True)
    s_perm = rank_to_score(r_perm, "perm_importance", descending=True)
    s_rfe = r_rfe[["feature", "rfe_selected"]].copy().rename(columns={"rfe_selected": "score"})

    pool = pd.DataFrame({"feature": X_enc.columns}).drop_duplicates()
    for name, s in [("cv", s_cv), ("mi", s_mi), ("rf", s_rf), ("perm", s_perm), ("rfe", s_rfe)]:
        pool = pool.merge(s.rename(columns={"score": f"score_{name}"}), on="feature", how="left")

    pool = pool.fillna(0)
    pool["score_consenso"] = (
        0.20 * pool["score_cv"] + 0.25 * pool["score_mi"] + 0.25 * pool["score_rf"] + 
        0.20 * pool["score_perm"] + 0.10 * pool["score_rfe"]
    )
    pool = pool.sort_values("score_consenso", ascending=False).reset_index(drop=True)

    yhat = rf.predict(Xte)
    f1m = f1_score(yte, yhat, average="macro")

    results[target] = {
        "cv": r_cv, "mi": r_mi, "rf": r_rf, "perm": r_perm, "rfe": r_rfe, "consenso": pool, "f1": f1m
    }
    
    print(f"Target: {target} completado. F1 Baseline: {f1m:.4f}")
    display(pool.head(15))
    
    # Limpiar antes del siguiente target
    del X_use, X_enc, Xtr, Xte, ytr, yte, rf, r_cv, r_mi, r_rf, r_perm, r_rfe, pool
    gc.collect()

print("\n=== EXPORTANDO RESULTADOS FINALES ===")
for target, data in results.items():
    fname = f"top30_features_{target}.csv"
    data["consenso"].head(30).to_csv(fname, index=False)
    print(f"Guardado: {fname}")



=== PROCESANDO TARGET: tipo_violencia ===
Preprocesando muestra (imputacion y limpieza)...
Calculando Cramers V...
Calculando Mutual Information...
Entrenando Random Forest para importancias...
Calculando Permutation Importance (top 40)...
Ejecutando RFECV (wrapper sobre Top 30 MI)...
Target: tipo_violencia completado. F1 Baseline: 0.9996


,feature,score_cv,score_mi,score_rf,score_perm,score_rfe,score_consenso
0,tipo_violencia_orig,1.000000,1.000000,1.000000,1.000000,1.0,1.000000
1,violacion,0.992248,0.976744,0.984496,0.992248,1.0,0.987209
2,gritos_insultos,0.984496,0.992248,0.992248,0.914729,1.0,0.975969
3,empujones,0.976744,0.961240,0.968992,0.976744,1.0,0.973256
4,punetazos,0.968992,0.953488,0.976744,0.984496,0.0,0.873256
5,bofetadas,0.930233,0.922481,0.922481,0.883721,0.0,0.824031
6,jalones_cabello,0.937984,0.930233,0.914729,0.837209,0.0,0.816279
7,edad_victima,0.899225,0.945736,0.899225,0.829457,0.0,0.806977
8,otra_vfis,0.883721,0.868217,0.930233,0.891473,0.0,0.804651
9,vinculo_pareja,0.852713,0.937984,0.891473,0.860465,0.0,0.800000



=== PROCESANDO TARGET: nivel_riesgo_victima ===
Preprocesando muestra (imputacion y limpieza)...
Calculando Cramers V...
Calculando Mutual Information...
Entrenando Random Forest para importancias...
Calculando Permutation Importance (top 40)...
Ejecutando RFECV (wrapper sobre Top 30 MI)...
Target: nivel_riesgo_victima completado. F1 Baseline: 0.9995


,feature,score_cv,score_mi,score_rf,score_perm,score_rfe,score_consenso
0,nivel_riesgo_victima_orig,1.000000,1.000000,1.000000,1.000000,1.0,1.000000
1,factor_agresor_consumo_alcohol,0.961240,0.953488,0.937984,0.992248,1.0,0.963566
2,amenaza_de_muerte,0.953488,0.937984,0.906977,0.984496,1.0,0.948837
3,tipo_violencia_orig,0.968992,0.968992,0.992248,0.457364,1.0,0.875581
4,ubigeo_nombre,0.976744,0.984496,0.976744,0.333333,1.0,0.852326
5,desea_denunciar,0.837209,0.922481,0.728682,0.968992,0.0,0.774031
6,ubigeo_codigo,0.984496,0.976744,0.968992,0.449612,0.0,0.773256
7,cem,0.992248,0.992248,0.984496,0.341085,0.0,0.760853
8,factor_agresor_consume_droga,0.945736,0.961240,0.868217,0.403101,0.0,0.727132
9,estado_agresor_g,0.922481,0.860465,0.837209,0.480620,0.0,0.705039
